# 09 · Results export for the offline dashboard (Stage 1; CPU only, read-only on the experiment)

Exports the **HEADPATH** results (notebook 08) and, separately, the **notebook-07 collapse experiment** context into
one validated bundle for the offline dashboard (`docs/DASHBOARD_PLAN.md`, `docs/DASHBOARD_EXPORT.md`).

* **Read-only.**
  * The exporter only reads `reports/`, `runs/headpath/`, `runs/collapse_diag/` and the 12 processed validation
    samples. `checkpoints/last.pt` is read for the validity checks only.
  * Every source file is content-hashed before and after; any change fails the export.
  * Nothing is written under `runs/`, `reports/` or `processed/`.
* **Validated.**
  * Every stored HEADPATH outcome is re-derived from the CSVs, and the McNemar test from the stored categories.
  * Labels are checked against the processed samples, grid metrics are recomputed from the full prediction maps, and
    every exported array is decoded again and compared bit for bit.
  * The recorded 0.10 vs implemented 0.08 / 0.12 P2 threshold inconsistency is **flagged**; the historical report
    is not modified.
* **No GPU, no training.** Runtime: a CPU runtime is enough. Estimate ≈ 10–25 min, mostly hashing about 1 GB of
  Drive files twice. Notebook 02 is not needed.
* **Output:** `MyDrive/VeriHGN/export/headpath_dashboard_<timestamp>.zip`, plus its integrity report (`.md` and
  `.json`). All values in the bundle come from the experiment files. This is validation data only; no test-set
  results exist.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, shutil, time
from pathlib import Path
from verihgn_repro.export import dashboard_bundle as DB
REPORT = '08_headpath_experiment.json'      # the authoritative HEADPATH report (other 08 versions are listed, not used)
CONTEXT07_REPORT = None                     # None: the notebook-07 report named in reports/07_forensic_audit.json
try:
    LOCAL_EXPORT_ROOT
except NameError:
    LOCAL_EXPORT_ROOT = '/content/dashboard_export'    # local VM disk: the bundle is built here, then copied to Drive
out = Path(LOCAL_EXPORT_ROOT) / f'headpath_dashboard_{time.strftime("%Y%m%d_%H%M%S")}'
t0 = time.time()
try:
    res = DB.export(L.root, out, report_name=REPORT, context07_report=CONTEXT07_REPORT)
except DB.ExportValidationError as e:
    failed_md = out / 'INTEGRITY_REPORT.md'
    if failed_md.exists():                  # keep the evidence on Drive (export/ only), then stop
        for f, suffix in ((failed_md, '_FAILED_INTEGRITY_REPORT.md'), (out / 'integrity_report.json', '_FAILED_integrity_report.json')):
            DB.copy_verified(f, L.export, name=out.name + suffix)
        print(failed_md.read_text()[:8000])
    raise
print(f"export {res['status']} in {time.time() - t0:.0f}s | checks {res['summary']} | zip {res['zip_bytes'] / 1e6:.1f} MB")
print(Path(res['integrity_report']).read_text()[:8000])

In [ ]:
# Copy the zip and its integrity report to Drive (export/ only), verify the copies, then flush Drive (last step).
from verihgn_repro import colab
copies = [DB.copy_verified(res['zip'], L.export),
          DB.copy_verified(out / 'INTEGRITY_REPORT.md', L.export, name=out.name + '_INTEGRITY_REPORT.md'),
          DB.copy_verified(out / 'integrity_report.json', L.export, name=out.name + '_integrity_report.json')]
v = DB.verify_bundle(copies[0]['path'])
assert v['status'] == 'PASSED', v
print(json.dumps({'zip': copies[0], 'verify_on_drive': v}, indent=1))
persistence = colab.flush_drive([c['path'] for c in copies])        # LAST: nothing is written to Drive after this
print('drive persistence:', persistence)
print('\n→ Download', Path(copies[0]['path']).name, '(sha256', copies[0]['sha256'][:16] + '…) and',
      Path(copies[1]['path']).name, 'from MyDrive/VeriHGN/export/, and share the integrity report.')